# 1. Lectura


In [0]:
spark.read.table("nadaq_raw")
#o
spark.table("nadaq_raw")

# 2. Creación de Feature


In [0]:
from pyspark.sql import functions as F, Window

#Como spark es por cluster hay que ordenar por que se trabajan con trozos distintos de los datos sin orden
df = spark.read.table("nadaq_raw")
# 2. Truco para clúster: Creamos una columna fija con el nombre del activo 
# para poder particionar y que Spark no agrupe todo en una sola máquina
df = df.withColumn("Asset", F.lit("NASDAQ"))
ventana_cronologica = Window.partitionBy("Asset").orderBy("Date")
#Retorno diario
df_transformado = (df
    .withColumn("Precio_dia_Anterior", F.lag("Close").over(ventana_cronologica))
    .withColumn("Retorno_diario", (F.col("Close")-F.col("Precio_dia_Anterior")) / F.col("Precio_dia_Anterior"))

)
df_transformado = df_transformado.dropna()
df_transformado.select("Date", "Close", "Precio_dia_Anterior","Retorno_diario").display()

# Guardar la tabla 

In [0]:
nombre_tabla = "nasdaq_retorno_raw"
df_transformado.write.format("delta").mode("overwrite").saveAsTable(nombre_tabla)
print("Datos guardados en:" + nombre_tabla)

In [0]:
spark.read.table("nasdaq_retorno_raw").display()